In [24]:
# Este script simula o "processo de pensamento" do agente, usando as ferramentas do
# Scikit-learn para tomar decisões inteligentes ao construir um pipeline.
# Cada passo é documentado com print para que você possa acompanhar o raciocínio.

### Script de Simulação do Agente Exploratório

# --- 1. Setup do Ambiente e Dados de Exemplo ---
import pandas as pd
import numpy as np

# Ferramentas de Introspecção e Validação do Scikit-learn
from sklearn.base import (
    is_classifier,
    is_regressor,
    is_outlier_detector,
    check_is_fitted,
    clone,
)
from sklearn.utils.validation import (
    check_X_y,
    check_array,
    check_is_fitted,
    has_fit_parameter,
    check_consistent_length,
    check_memory,
    check_non_negative,
    check_random_state,
    check_scalar,
    check_symmetric,
    check_array,
    check_is_fitted,
    NotFittedError,
)
from sklearn.utils.multiclass import type_of_target

# Estimadores de Exemplo para o Teste
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.svm import SVC


In [25]:

print("--- Script de Simulação do Agente Iniciado ---")
print("Carregando ferramentas e dados de exemplo...")

# Dados de exemplo para uma tarefa de CLASSIFICAÇÃO
X_clf_data = pd.DataFrame({
    'feature1': np.random.rand(100) * 10,
    'feature2': np.random.rand(100) * 5,
})
y_clf_data = pd.Series(np.random.choice([0, 1, 2], 100))


--- Script de Simulação do Agente Iniciado ---
Carregando ferramentas e dados de exemplo...


In [17]:
# --- 2. O Agente Analisa a Tarefa ---
print("\n--- PASSO 1: Análise da Tarefa ---")
task_type = type_of_target(y_clf_data)
print(f"Agente: Analisei a coluna alvo 'y'. O Scikit-learn me informou que o tipo é: '{task_type}'.")

if task_type in ('binary', 'multiclass'):
    print("Agente: DECISÃO -> A tarefa é de CLASSIFICAÇÃO. Vou procurar por classificadores.")
    agent_task = 'classification'
elif task_type == 'continuous':
    print("Agente: DECISÃO -> A tarefa é de REGRESSÃO. Vou procurar por regressores.")
    agent_task = 'regression'
else:
    print(f"Agente: DECISÃO -> Tarefa não supervisionada ou desconhecida ('{task_type}').")
    agent_task = 'unsupervised'



--- PASSO 1: Análise da Tarefa ---
Agente: Analisei a coluna alvo 'y'. O Scikit-learn me informou que o tipo é: 'multiclass'.
Agente: DECISÃO -> A tarefa é de CLASSIFICAÇÃO. Vou procurar por classificadores.


In [26]:
# --- 3. O Agente Seleciona Estimadores Compatíveis ---
print("\n--- PASSO 2: Seleção de Estimadores (Poda da Árvore de Busca) ---")
possible_estimators = [
    RandomForestClassifier(),
    RandomForestRegressor(),
    SVC(),
    StandardScaler() # Um transformador, que não é classificador nem regressor
]
print(f"Agente: Tenho uma lista de {len(possible_estimators)} estimadores candidatos.")

valid_estimators = []
if agent_task == 'classification':
    for estimator in possible_estimators:
        if is_classifier(estimator):
            valid_estimators.append(estimator)
            print(f"  - Verificando '{estimator.__class__.__name__}': É um classificador. OK.")
        else:
            print(f"  - Verificando '{estimator.__class__.__name__}': NÃO é um classificador. Descartando.")

print(f"Agente: DECISÃO -> Minha lista de candidatos foi reduzida para {len(valid_estimators)} estimador(es) compatível(is).")
final_estimator = valid_estimators[0]
print(f"Agente: Vou explorar o primeiro da lista: '{final_estimator.__class__.__name__}'.")




--- PASSO 2: Seleção de Estimadores (Poda da Árvore de Busca) ---
Agente: Tenho uma lista de 4 estimadores candidatos.
  - Verificando 'RandomForestClassifier': É um classificador. OK.
  - Verificando 'RandomForestRegressor': NÃO é um classificador. Descartando.
  - Verificando 'SVC': É um classificador. OK.
  - Verificando 'StandardScaler': NÃO é um classificador. Descartando.
Agente: DECISÃO -> Minha lista de candidatos foi reduzida para 2 estimador(es) compatível(is).
Agente: Vou explorar o primeiro da lista: 'RandomForestClassifier'.


In [21]:

# --- 4. O Agente Verifica a Compatibilidade de Parâmetros Avançados ---
print("\n--- PASSO 3: Verificação de Compatibilidade de Parâmetros ---")
print("Agente: Estou pensando em usar pesos nas amostras ('sample_weight') para lidar com desbalanceamento.")
print(f"Agente: Vou perguntar ao '{final_estimator.__class__.__name__}' se ele aceita o parâmetro 'sample_weight' no seu método fit.")

if has_fit_parameter(final_estimator, "sample_weight"):
    print(f"Agente: DECISÃO -> Sim, '{final_estimator.__class__.__name__}' aceita 'sample_weight'. Posso explorar essa estratégia.")
else:
    print(f"Agente: DECISÃO -> Não, '{final_estimator.__class__.__name__}' não aceita 'sample_weight'. Vou pular essa otimização e economizar tempo.")




--- PASSO 3: Verificação de Compatibilidade de Parâmetros ---
Agente: Estou pensando em usar pesos nas amostras ('sample_weight') para lidar com desbalanceamento.
Agente: Vou perguntar ao 'RandomForestClassifier' se ele aceita o parâmetro 'sample_weight' no seu método fit.
Agente: DECISÃO -> Sim, 'RandomForestClassifier' aceita 'sample_weight'. Posso explorar essa estratégia.


In [22]:
# --- 5. O Agente Valida os Passos do Pipeline Durante a Construção ---
print("\n--- PASSO 4: Validação da Integridade dos Dados no Pipeline ---")
print("Agente: Estou tentando construir um pipeline. Vou testar um transformador 'malicioso' que gera NaNs.")

def faulty_transformer_func(X):
    X_transformed = X.copy()
    # Introduzindo um NaN de propósito
    X_transformed.iloc[0, 0] = np.nan
    return X_transformed

faulty_transformer = FunctionTransformer(faulty_transformer_func)

try:
    print("  - Agente: Aplicando o transformador 'malicioso'...")
    X_faulty = faulty_transformer.fit_transform(X_clf_data)
    print("  - Agente: Agora, vou validar a saída com check_array antes de passar para o próximo passo...")
    # Por padrão, check_array não permite NaNs. Isso vai gerar um erro.
    check_array(X_faulty)
    print("  - Agente: A validação passou (isso não deveria acontecer).")
except ValueError as e:
    print(f"Agente: DECISÃO -> A validação falhou! Erro: '{e}'.")
    print("  - Agente: Este 'ramo' da minha árvore de exploração produziu dados inválidos. Vou descartá-lo e tentar outro caminho (ex: outro imputer ou nenhum).")



--- PASSO 4: Validação da Integridade dos Dados no Pipeline ---
Agente: Estou tentando construir um pipeline. Vou testar um transformador 'malicioso' que gera NaNs.
  - Agente: Aplicando o transformador 'malicioso'...
  - Agente: Agora, vou validar a saída com check_array antes de passar para o próximo passo...
Agente: DECISÃO -> A validação falhou! Erro: 'Input contains NaN.'.
  - Agente: Este 'ramo' da minha árvore de exploração produziu dados inválidos. Vou descartá-lo e tentar outro caminho (ex: outro imputer ou nenhum).


In [23]:
# --- 6. O Agente Explora Variações Usando clone ---
print("\n--- PASSO 5: Exploração de Variações com clone ---")
base_model = RandomForestClassifier(n_estimators=100, random_state=42)
print(f"Agente: Criei um modelo base: {base_model}")

try:
    check_is_fitted(base_model)
except NotFittedError:
    print("  - Agente: Verifiquei o modelo base com check_is_fitted. Ele ainda não foi treinado. Correto.")

print("\nAgente: Agora, quero testar uma variação com mais árvores sem alterar meu modelo base.")
# Clonando o modelo. O clone tem os mesmos parâmetros, mas não está treinado.
model_clone = clone(base_model)
model_clone.set_params(n_estimators=200)
print(f"Agente: Criei um clone e alterei seus parâmetros: {model_clone}")

print("\nAgente: Vou treinar apenas o clone...")
model_clone.fit(X_clf_data, y_clf_data)
print("  - Agente: Clone treinado com sucesso.")

print("\nAgente: Verificando o estado dos modelos agora:")
try:
    check_is_fitted(base_model)
except NotFittedError:
    print("  - Modelo Base: Continua não treinado. Perfeito, clone evitou o efeito colateral.")

try:
    check_is_fitted(model_clone)
    print("  - Modelo Clone: Está treinado. check_is_fitted confirmou.")
except NotFittedError:
    print("  - Modelo Clone: Não está treinado (isso não deveria acontecer).")

print("\n--- Simulação Concluída ---")


--- PASSO 5: Exploração de Variações com clone ---
Agente: Criei um modelo base: RandomForestClassifier(random_state=42)
  - Agente: Verifiquei o modelo base com check_is_fitted. Ele ainda não foi treinado. Correto.

Agente: Agora, quero testar uma variação com mais árvores sem alterar meu modelo base.
Agente: Criei um clone e alterei seus parâmetros: RandomForestClassifier(n_estimators=200, random_state=42)

Agente: Vou treinar apenas o clone...
  - Agente: Clone treinado com sucesso.

Agente: Verificando o estado dos modelos agora:
  - Modelo Base: Continua não treinado. Perfeito, clone evitou o efeito colateral.
  - Modelo Clone: Está treinado. check_is_fitted confirmou.

--- Simulação Concluída ---
